# Backpropagation, by Hand: The Chain Rule, Run Backward

The runnable code of [Backpropagation, by Hand: The Chain Rule, Run Backward](https://residual-stream.lvtan71.workers.dev/posts/backpropagation-by-hand/), in the order the post runs it. Each
code cell is followed by the output the post printed, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/backpropagation-by-hand.ipynb)

**Hardware:** Runs on a CPU; Colab's free runtime is enough. About 4 min end to end.


The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
# Setup: fetch the helpers and data this post's code uses into a working folder.
import os, shutil, subprocess, sys
SLUG = "backpropagation-by-hand"
ROOT = os.path.abspath("..") if os.path.isdir("../shared") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
WORK = os.path.join(ROOT, "work", SLUG)
os.makedirs(WORK, exist_ok=True)
for f in ['shared/backpropagation-by-hand/mlp.py', 'shared/features-overfitting-regularization/overfit.py', 'shared/features-overfitting-regularization/router_data.py', 'shared/measuring-a-classifier/router_models.py', 'data/router-questions.csv']:
    shutil.copy(os.path.join(ROOT, f), WORK)
    if f.endswith(".tar.gz"):                       # the blog's posts, as of that run
        shutil.unpack_archive(os.path.join(WORK, os.path.basename(f)), WORK)
os.chdir(WORK)
sys.path.insert(0, WORK)
print("working in", WORK)

## One step, every number

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/backpropagation-by-hand/#one-step-every-number)

In [ ]:
%%writefile mlp.py
import numpy as np

sigmoid = lambda z: 1 / (1 + np.exp(-z))
relu = lambda z: np.maximum(z, 0.0)

def init(n_in, n_hidden, rng):
    """He initialization for the ReLU layer; small random output weights."""
    return {"W1": rng.normal(0, np.sqrt(2 / n_in), (n_in, n_hidden)),
            "b1": np.zeros(n_hidden),
            "w2": rng.normal(0, np.sqrt(1 / n_hidden), n_hidden),
            "b2": 0.0}

def forward(P, X):
    z1 = X @ P["W1"] + P["b1"]            # (n, hidden)
    a1 = relu(z1)
    z2 = a1 @ P["w2"] + P["b2"]           # (n,)  one logit per example
    return z1, a1, z2

def backward(P, X, y, lam=0.0):
    """Gradients of the mean log-loss (+ L2 on weights) for every parameter."""
    n = len(y)
    z1, a1, z2 = forward(P, X)
    p = sigmoid(z2)
    dz2 = (p - y) / n                     # the p - y of "Learning Is Optimization", per example
    grads = {"w2": a1.T @ dz2 + lam * P["w2"],
             "b2": dz2.sum()}
    da1 = np.outer(dz2, P["w2"])          # back through the output weights
    dz1 = da1 * (z1 > 0)                  # back through ReLU: pass or block
    grads["W1"] = X.T @ dz1 + lam * P["W1"]
    grads["b1"] = dz1.sum(axis=0)
    p = np.clip(p, 1e-12, 1 - 1e-12)
    loss = -(y * np.log(p) + (1 - y) * np.log(1 - p)).mean()
    loss += 0.5 * lam * ((P["W1"] ** 2).sum() + (P["w2"] ** 2).sum())
    return grads, loss

def train(X, y, n_hidden, lam=0.0, steps=3000, eta=0.5, seed=0):
    P = init(X.shape[1], n_hidden, np.random.default_rng(seed))
    for _ in range(steps):
        grads, _ = backward(P, X, y, lam)
        for k in grads:
            P[k] = P[k] - eta * grads[k]
    return P

In [ ]:
import numpy as np
from mlp import sigmoid, relu, forward, backward, train

X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y = np.array([0, 1, 1, 0], dtype=float)          # XOR

# One example, one step, every number.
P = {"W1": np.array([[0.5, -0.4], [0.3, 0.8]]), "b1": np.array([0.0, 0.1]),
     "w2": np.array([0.7, -0.6]), "b2": 0.1}
x, t = X[2:3], y[2:3]                             # x = (1, 0), target 1
z1, a1, z2 = forward(P, x)
grads, loss = backward(P, x, t)
print(f"z1 {z1[0]}  a1 {a1[0]}  z2 {z2[0]:.4f}  p {sigmoid(z2[0]):.4f}  loss {loss:.4f}")
for k in ["w2", "b2", "W1", "b1"]:
    print(f"dL/d{k} = {np.round(grads[k], 4).tolist()}")
P = {k: v - 0.5 * grads[k] for k, v in P.items()}
print(f"after one step: p {sigmoid(forward(P, x)[2][0]):.4f}  loss {backward(P, x, t)[1]:.4f}")

# How often does training find XOR, by hidden-layer width?
for h in [2, 3, 4, 8, 16]:
    solved = 0
    for seed in range(100):
        P = train(X, y, h, steps=5000, eta=1.0, seed=seed)
        solved += ((sigmoid(forward(P, X)[2]) >= 0.5) == y).all()
    print(f"{h:2d} hidden units: XOR solved from {solved}/100 random starts")

The post printed:

```text
z1 [ 0.5 -0.3]  a1 [0.5 0. ]  z2 0.4500  p 0.6106  loss 0.4932
dL/dw2 = [-0.1947, 0.0]
dL/db2 = -0.3894
dL/dW1 = [[-0.2726, 0.0], [0.0, 0.0]]
dL/db1 = [-0.2726, 0.0]
after one step: p 0.7131  loss 0.3381
 2 hidden units: XOR solved from 18/100 random starts
 3 hidden units: XOR solved from 52/100 random starts
 4 hidden units: XOR solved from 73/100 random starts
 8 hidden units: XOR solved from 96/100 random starts
16 hidden units: XOR solved from 100/100 random starts
```

## Can training find it?

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/backpropagation-by-hand/#can-training-find-it)

In [ ]:
import numpy as np
from collections import Counter
from mlp import sigmoid, forward, backward, init, train

X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y = np.array([0, 1, 1, 0], dtype=float)

# Why do 82 of 100 two-unit starts fail? Classify every failure.
outcome, dead_start, solved_if_dead = Counter(), 0, 0
for seed in range(100):
    W1 = init(2, 2, np.random.default_rng(seed))["W1"]
    dead = ((W1 <= 0).all(axis=0)).any()          # a unit with both weights <= 0 never fires
    dead_start += dead
    P = train(X, y, 2, steps=5000, eta=1.0, seed=seed)
    right = int(((sigmoid(forward(P, X)[2]) >= 0.5) == y).sum())
    solved_if_dead += dead and right == 4
    if right < 4:
        grads, loss = backward(P, X, y)
        size = np.sqrt(sum((g ** 2).sum() for g in grads.values()))
        outcome[(right, round(loss, 3), "dead unit at start" if dead else "both units alive",
                 "gradient < 1e-3" if size < 1e-3 else "gradient >= 1e-3")] += 1
print(f"starts with a unit dead from step 0: {dead_start}/100, of which solved: {solved_if_dead}")
for (right, loss, dead, grad), n in sorted(outcome.items(), key=lambda kv: -kv[1]):
    print(f"{n:3d} failures: {right}/4 right, loss {loss:.3f}, {dead}, {grad}")

The post printed:

```text
starts with a unit dead from step 0: 42/100, of which solved: 0
 27 failures: 3/4 right, loss 0.477, dead unit at start, gradient < 1e-3
 22 failures: 3/4 right, loss 0.347, both units alive, gradient < 1e-3
 15 failures: 2/4 right, loss 0.693, dead unit at start, gradient < 1e-3
 14 failures: 3/4 right, loss 0.477, both units alive, gradient < 1e-3
  2 failures: 2/4 right, loss 0.693, both units alive, gradient < 1e-3
  1 failures: 3/4 right, loss 0.347, both units alive, gradient >= 1e-3
  1 failures: 3/4 right, loss 0.693, both units alive, gradient < 1e-3
```

## Checking the gradients

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/backpropagation-by-hand/#checking-the-gradients)

In [ ]:
import numpy as np
from mlp import init, backward

# Compare backprop to finite differences on a small random problem.
rng = np.random.default_rng(1)
X, y = rng.normal(size=(5, 4)), rng.integers(0, 2, 5).astype(float)
P = init(4, 3, rng)
grads, _ = backward(P, X, y, lam=0.01)
eps, worst = 1e-6, 0.0
for k in ["W1", "b1", "w2"]:
    for i in np.ndindex(P[k].shape):
        old = P[k][i]
        P[k][i] = old + eps; up = backward(P, X, y, 0.01)[1]
        P[k][i] = old - eps; down = backward(P, X, y, 0.01)[1]
        P[k][i] = old
        numeric = (up - down) / (2 * eps)                 # central difference
        worst = max(worst, abs(numeric - grads[k][i]) / max(1e-12, abs(numeric) + abs(grads[k][i])))
print(f"largest relative error, backprop vs finite differences: {worst:.1e}")

The post printed:

```text
largest relative error, backprop vs finite differences: 9.7e-08
```

## The router, with a hidden layer

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/backpropagation-by-hand/#the-router-with-a-hidden-layer)

In [ ]:
import numpy as np
from mlp import sigmoid, forward, backward, train
from overfit import log_loss, train as train_linear
from router_data import split, featurize
from router_models import keyword_count

def features(rs, X):
    return np.c_[X, keyword_count(rs)]            # bag of words + keyword count

models = {"linear, L2 0.003": None, "linear, L2 0.01": None}
models.update({f"16 hidden, L2 {lam}": lam for lam in [0.0, 0.003, 0.01, 0.03, 0.1]})
val, test, train_loss = {m: [] for m in models}, {m: [] for m in models}, {m: [] for m in models}
for seed in range(20):
    tr, va, te = split(seed)
    ((Xtr, ytr), (Xva, yva), (Xte, yte)), _ = featurize(tr, va, te)
    A, B, C = features(tr, Xtr), features(va, Xva), features(te, Xte)
    for name, lam in models.items():
        if name.startswith("linear"):
            w, b = train_linear(A, ytr, float(name.split()[-1]))
            prob = lambda M: sigmoid(M @ w + b)
        else:
            P = train(A, ytr, n_hidden=16, lam=lam, seed=seed)
            prob = lambda M: sigmoid(forward(P, M)[2])
        for rows, M, t in [(val, B, yva), (test, C, yte), (train_loss, A, ytr)]:
            p = prob(M)
            rows[name].append((log_loss(p, t), ((p >= .5) == t).mean()))

corrected = np.sqrt(1 / 20 + 40 / 120)            # Nadeau-Bengio, "Features and Overfitting"
base = np.array(val["linear, L2 0.003"])
print(f"{'':20s} {'train':>6s} {'val loss':>9s} {'vs linear 0.003':>16s} {'val acc':>8s}")
for name in models:
    r, t = np.array(val[name]), np.array(train_loss[name])
    d = r[:, 0] - base[:, 0]
    vs = f"{d.mean():+.3f} ± {d.std(ddof=1) * corrected:.3f}" if d.any() else ""
    print(f"{name:20s} {t[:, 0].mean():6.3f} {r[:, 0].mean():9.3f} {vs:>16s} {r[:, 1].mean():8.3f}")

best = min((m for m in models if m.startswith("16")), key=lambda m: np.mean(np.array(val[m])[:, 0]))
a, b = np.array(test[best])[:, 0], np.array(test["linear, L2 0.003"])[:, 0]
print(f"\nchosen on validation: {best}. test loss {a.mean():.3f} vs linear {b.mean():.3f}, "
      f"difference {(a - b).mean():+.3f} ± {(a - b).std(ddof=1) * corrected:.3f}")

The post printed:

```text
                      train  val loss  vs linear 0.003  val acc
linear, L2 0.003      0.065     0.347                     0.866
linear, L2 0.01       0.141     0.346   -0.001 ± 0.021    0.861
16 hidden, L2 0.0     0.000     0.689   +0.343 ± 0.143    0.872
16 hidden, L2 0.003   0.010     0.416   +0.069 ± 0.040    0.868
16 hidden, L2 0.01    0.032     0.365   +0.018 ± 0.015    0.869
16 hidden, L2 0.03    0.098     0.342   -0.005 ± 0.011    0.866
16 hidden, L2 0.1     0.314     0.399   +0.053 ± 0.047    0.834

chosen on validation: 16 hidden, L2 0.03. test loss 0.332 vs linear 0.333, difference -0.002 ± 0.008
```

In [ ]:
import numpy as np
from mlp import sigmoid, forward, train
from overfit import log_loss
from router_data import split, featurize
from router_models import keyword_count

# Does the answer depend on the step count or on the initialization seed?
runs = {}
for seed in range(20):
    tr, va, te = split(seed)
    ((Xtr, ytr), (Xva, yva), _), _ = featurize(tr, va, te)
    A, B = np.c_[Xtr, keyword_count(tr)], np.c_[Xva, keyword_count(va)]
    for lam in [0.0, 0.03]:
        for steps in [3000, 12000]:
            for init_seed in range(5):
                P = train(A, ytr, n_hidden=16, lam=lam, steps=steps, seed=100 * init_seed + seed)
                runs.setdefault((lam, steps, init_seed), []).append(
                    log_loss(sigmoid(forward(P, B)[2]), yva))
for lam in [0.0, 0.03]:
    for steps in [3000, 12000]:
        means = [np.mean(runs[(lam, steps, s)]) for s in range(5)]
        print(f"L2 {lam:<4} {steps:6,d} steps: val loss {min(means):.3f}-{max(means):.3f} "
              f"across 5 initialization seeds")

The post printed:

```text
L2 0.0   3,000 steps: val loss 0.677-0.692 across 5 initialization seeds
L2 0.0  12,000 steps: val loss 0.778-0.795 across 5 initialization seeds
L2 0.03  3,000 steps: val loss 0.342-0.343 across 5 initialization seeds
L2 0.03 12,000 steps: val loss 0.342-0.342 across 5 initialization seeds
```